### Importing necessary libraries

In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

##Tensor flow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import Input
from tensorflow.keras.optimizers import Adam


#### Data Preparation

In [36]:
df = pd.read_csv('../data/coffe_quality_clean.csv')

In [37]:
X = df.drop(columns=["Total Cup Points"])
y = df["Total Cup Points"]

#### Train-Test Split

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [39]:
## Data Scalling
x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1,1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1,1))

#### Creation of The Model

In [40]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(1)
]
)

In [41]:
model.compile(
    optimizer = Adam(learning_rate=0.001),
    loss = "mse",
    metrics=["mae"]
)

#### Model Training

In [42]:
history = model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=100,
    validation_split=0.2,
    verbose =1
)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.8472 - mae: 0.7061 - val_loss: 0.7687 - val_mae: 0.6538
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.7169 - mae: 0.6456 - val_loss: 0.6501 - val_mae: 0.5966
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.6094 - mae: 0.5897 - val_loss: 0.5368 - val_mae: 0.5369
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.5160 - mae: 0.5348 - val_loss: 0.4424 - val_mae: 0.4860
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.4255 - mae: 0.4802 - val_loss: 0.3649 - val_mae: 0.4436
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.3429 - mae: 0.4310 - val_loss: 0.2976 - val_mae: 0.4028
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2745 - mae: 0.3827 - val_loss: 0.2403 - val_mae: 0.3624
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2202 - mae: 0.3421 - val_loss: 0.1945 - val_mae: 0.3269
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1735 - mae: 

In [43]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 32)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,789 (10.90 KB)

 Trainable params: 929 (3.63 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,860 (7.27 KB)

In [44]:
## MAE
pred_scaled = model.predict(X_test_scaled)

pred = y_scaler.inverse_transform(pred_scaled).flatten()

from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_test, pred)
print("mae= ", mae)

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7f26f42e8180> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
mae=  0.11687859671456481


## Results

In [45]:
pred = pred.flatten()
pred_real = y_scaler.inverse_transform(pred_scaled).flatten()
for real, estimado in zip(y_test[:10], pred_real[:10]):
    print(f"Real: {real:.2f} | prediction: {estimado:.2f}")

Real: 82.50 | prediction: 82.27
Real: 86.08 | prediction: 86.24
Real: 84.42 | prediction: 84.36
Real: 83.92 | prediction: 83.96
Real: 82.33 | prediction: 82.29
Real: 86.50 | prediction: 86.31
Real: 83.83 | prediction: 83.47
Real: 83.17 | prediction: 83.08
Real: 85.92 | prediction: 85.92
Real: 82.75 | prediction: 82.44


#             Thank you for checking out this project!! :D